# Интерполяция видео до 60 FPS — блокнот для Kaggle

Готовая последовательность ячеек для запуска на Kaggle (включите GPU и Internet):
- Проверка окружения
- Smoke‑test ffmpeg (2s)
- Опционально: RIFE (DL)
- Опционально: Real-ESRGAN (апскейл)

Загрузите `dn.mp4` в `/kaggle/working/input/` перед запуском ячеек.

In [ ]:
# Ячейка 1 — проверка окружения (Python, torch, ffmpeg, nvidia)
import sys, subprocess, shutil
print('Python:', sys.version.split('\n')[0])
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
except Exception as e:
    print('torch import error:', e)
if shutil.which('ffmpeg'):
    p = subprocess.run(['ffmpeg','-version'], stdout=subprocess.PIPE, text=True)
    print('ffmpeg:', p.stdout.splitlines()[0])
else:
    print('ffmpeg not found in PATH')
# nvidia-smi optional
try:
    p = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print('nvidia-smi lines:', len(p.stdout.splitlines()))
except Exception as e:
    print('nvidia-smi not available:', e)


In [ ]:
# Ячейка 2 — параметры
# Automatically choose input from mounted Kaggle dataset, working input, or repository test.mp4
import os
# Paths to check (Kaggle datasets are usually mounted under /kaggle/input/<dataset-name>/)
KAGGLE_DATASET_PATH = '/kaggle/input/testmp4/test.mp4'
WORKING_INPUT_PATH = '/kaggle/working/input/test.mp4'
REPO_TEST_PATH = os.path.join(os.getcwd(), 'test.mp4')
OUTPUT_DIR = '/kaggle/working/output'
# Ensure working input and output directories exist
os.makedirs('/kaggle/working/input', exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Select the first existing input file
if os.path.exists(KAGGLE_DATASET_PATH):
    INPUT = KAGGLE_DATASET_PATH
elif os.path.exists(WORKING_INPUT_PATH):
    INPUT = WORKING_INPUT_PATH
elif os.path.exists(REPO_TEST_PATH):
    INPUT = REPO_TEST_PATH
else:
    # Fallback to original default path
    INPUT = '/kaggle/working/input/dn.mp4'
print('INPUT =', INPUT)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
%%bash
set -e
echo 'Smoke-test: 2s minterpolate -> /kaggle/working/output/dn_60_short.mp4'
mkdir -p /kaggle/working/output
if [ ! -f /kaggle/working/input/dn.mp4 ]; then echo 'Upload dn.mp4 to /kaggle/working/input/ first' >&2; exit 1; fi
ffmpeg -y -t 2 -i /kaggle/working/input/dn.mp4 -vf "minterpolate=mi_mode=mci:mc_mode=aobmc:vsbmc=1:fps=60" -c:v libx264 -preset veryfast -crf 20 /kaggle/working/output/dn_60_short.mp4
ls -lh /kaggle/working/output/dn_60_short.mp4 || true


## RIFE (опционально) — DL-интерполяция
Если хотите лучшее качество, запустите RIFE. Требует веса модели и GPU.

In [ ]:
%%bash
set -e
if [ ! -d /kaggle/working/arXiv2019-RIFE ]; then
  git clone https://github.com/hzwer/arXiv2019-RIFE.git /kaggle/working/arXiv2019-RIFE
fi
pip install -q -r /kaggle/working/arXiv2019-RIFE/requirements.txt || true
echo 'Upload RIFE weights into /kaggle/working/arXiv2019-RIFE/train_log per repo README'
if [ -f /kaggle/working/arXiv2019-RIFE/inference_video.py ]; then
  python /kaggle/working/arXiv2019-RIFE/inference_video.py --exp=slomo --video /kaggle/working/input/dn.mp4 --output /kaggle/working/output/dn_60fps_rife.mp4 --fp16 || true
else
  echo 'RIFE script not found' >&2
fi
ls -lh /kaggle/working/output || true


## Real-ESRGAN (апскейл) — опционально
Пример: извлечь кадры из 60fps видео, апскейлить и собрать обратно.

In [ ]:
%%bash
set -e
if [ ! -d /kaggle/working/Real-ESRGAN ]; then
  git clone https://github.com/xinntao/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
fi
pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt || true
mkdir -p /kaggle/working/frames /kaggle/working/upscaled_frames
if [ ! -f /kaggle/working/output/dn_60fps_rife.mp4 ]; then echo 'Put 60fps RIFE output in /kaggle/working/output/' >&2; fi
ffmpeg -y -i /kaggle/working/output/dn_60fps_rife.mp4 -vsync 0 /kaggle/working/frames/frame_%08d.png || true
python /kaggle/working/Real-ESRGAN/inference_realesrgan.py --input /kaggle/working/frames --output /kaggle/working/upscaled_frames --model RealESRGAN_x4plus --tile 400 --tile_pad 10 --fp16 || true
ffmpeg -y -framerate 60 -i /kaggle/working/upscaled_frames/frame_%08d_up.png -c:v libx264 -pix_fmt yuv420p /kaggle/working/output/dn_60fps_4k_noaudio.mp4 || true
ffmpeg -y -i /kaggle/working/output/dn_60fps_4k_noaudio.mp4 -i /kaggle/working/input/dn.mp4 -c copy -map 0:v:0 -map 1:a:0 /kaggle/working/output/dn_60fps_4k_with_audio.mp4 || true
ls -lh /kaggle/working/output || true


### Команды для Windows (локально)
Примеры: ремукс, smoke‑test и полная конвертация в 60fps (ffmpeg).

In [ ]:
# Локальные команды:
# cd /d D:\PHPStormProjects\interup
# ffprobe -v quiet -print_format json -show_format -show_streams dn.mp4
# ffmpeg -y -t 2 -i "dn.mp4" -vf "minterpolate=mi_mode=mci:mc_mode=aobmc:vsbmc=1:fps=60" -c:v libx264 -preset veryfast -crf 20 dn_60_short.mp4
print('See markdown cell for Windows commands')
